# FairHire-ES: Spanish Resume Screening Fairness Benchmark

**Author:** Alex Ramírez ([davidramsem](https://github.com/davidramsem)) — AI for HR/recruitment, Mexico City  
**Challenge:** [DEV × Kaggle Benchmarking Challenge](https://dev.to/challenges/kaggle-2026-09-23)  
**Data:** 100% **SYNTHETIC** (no real candidates)

This notebook:
1. Loads the synthetic FairHire-ES dataset
2. Runs local deterministic baselines (oracle / keyword / biased / noisy)
3. Scores accuracy + skill F1 + demographic twin fairness
4. Writes figures and a leaderboard

The Kaggle-ready task lives in `src/fairhire_es_kaggle_task.py` (push with `kaggle b t push`).


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = Path('/workspace/kaggle_challenge')
sys.path.insert(0, str(ROOT / 'src'))
print('ROOT =', ROOT)
print('cases exist:', (ROOT / 'data' / 'fairhire_es_cases.json').exists())


In [ ]:
import subprocess, sys
# Regenerate synthetic data + run baselines
subprocess.check_call([sys.executable, str(ROOT / 'src' / 'generate_dataset.py')])
subprocess.check_call([sys.executable, str(ROOT / 'src' / 'run_local.py')])
subprocess.check_call([sys.executable, str(ROOT / 'src' / 'validate_kaggle_task_local.py')])
print('Pipeline complete')


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image

lb = json.loads((ROOT / 'results' / 'local_leaderboard.json').read_text())
display(pd.DataFrame(lb)[[
    'model','composite_score','mae_fit_score','mean_required_f1','parse_rate'
]])
print('Fairness (mean twin spread):')
for r in lb:
    print(f"  {r['model']}: {r['fairness']['mean_twin_score_spread']}")


## Figures (from this run)

In [ ]:
from IPython.display import Image, display
for name in [
    '01_composite_scores.png',
    '02_accuracy_vs_fairness.png',
    '03_twin_spreads.png',
    '04_skill_f1.png',
]:
    p = ROOT / 'figures' / name
    print(name)
    display(Image(filename=str(p)))


## What the local baselines teach us

| Baseline | Role | Takeaway |
|---|---|---|
| `baseline_oracle` | Upper bound using gold skill overlap | Composite = 1.0 — scoring math is consistent |
| `baseline_keyword` | Regex skill detection on CV text | High F1, **0 twin spread** (demographics ignored) |
| `baseline_biased_name` | Same keywords + synthetic demographic penalties | Twin spread jumps (~15 pts) — harness **detects** unfairness |
| `baseline_noisy` | Overconfident + skill hallucinations | MAE and F1 collapse — scorer catches junk outputs |

**Next step on Kaggle:** push `src/fairhire_es_kaggle_task.py` and run against Gemini / Grok / etc. Compare whether frontier models behave more like `keyword` (fair) or leak demographic sensitivity like `biased_name`.


In [ ]:
val = json.loads((ROOT / 'results' / 'kaggle_task_local_validation.json').read_text())
print('Kaggle task local validation:', val)
assert val['validation'] == 'passed'
print('NOTEBOOK VERIFICATION: PASSED')
